In [ ]:
import json 
import pandas as pd 
import numpy as np 
import os 

RESULTS_DIR = "/home/david/Desktop/yuna/HPA/evaluation/logits/pretrained"
def get_confidence_result(model='llava-v1.6-vicuna-7b-hf', dataset='vqa_1k_control'): 
    
    result = [] 
    filepath = f'{RESULTS_DIR}/{model}/{dataset}.jsonl' 
    if not os.path.exists(filepath): 
        print(filepath, 'does not exist')
        return 
    with open(filepath) as f:
        data_list = [json.loads(l) for l in f]
    
    if len(data_list) == 0: 
        os.remove(filepath)
        print(f'deleted', filepath )
        return 

    if len(data_list) < 1000:
        print(f'skip MODEL: {model} | DATASET: {dataset} | loaded {len(data_list)}')  
        return result
    else: 
       print(f'MODEL: {model} | DATASET: {dataset} | loaded {len(data_list)}')  
 
    for example in data_list:  
        output_answer_key = 'generated_answers' if 'generated_answers' in example.keys() else 'answers' 
        # dict_keys(['image_id', 'question', 'question_id', 'deictic_removed', 'object_removed', 'weaker_object', 'subject_ablated', 'image', 'answers', 'generated_logits'])
        if 'generated_logits' not in example.keys(): 
            print('cannot process', example.keys(), example) 
            continue 
        for q in example['generated_logits'].keys():   
            logits = example['generated_logits'][q]['content']
            token_probs = np.exp([t['logprob'] for t in logits])
            confidence = token_probs.mean()  
            # print(f"{q}: {example[q]} \nAnswer: {example[output_answer_key][q]} conf: {confidence:.2f}")  
            result.append({
                'model': model, 
                'dataset': dataset, 
                'question_id': example['question_id'],
                'question': example[q], 
                'control_type': q, 
                'output': example[output_answer_key][q], 
                'confidence': round(confidence, 2)
            })
    return result  

In [34]:
models = os.listdir(RESULTS_DIR) # ["Qwen3-VL-8B-Instruct"]   # 'llava-v1.6-vicuna-7b-hf'] #  
datasets = ['vqa_1k_control', 'vqa_1k_control_inst_blind'] # 'vqa_1k', 'vqa_1k_inst_blind',  

df = []
for model in models: 
    for ds in datasets: 
        df.append(get_confidence_result(model, ds)) 
df = pd.DataFrame([item for sublist in df for item in (sublist if sublist else [])]) 
df.groupby(['model', 'dataset', 'control_type'])['output'].count()

skip MODEL: Qwen3-VL-32B-Instruct | DATASET: vqa_1k_control | loaded 43
skip MODEL: Qwen3-VL-32B-Instruct | DATASET: vqa_1k_control_inst_blind | loaded 18
MODEL: llava-v1.6-vicuna-7b-hf | DATASET: vqa_1k_control | loaded 1000
MODEL: llava-v1.6-vicuna-7b-hf | DATASET: vqa_1k_control_inst_blind | loaded 1000
MODEL: llava-v1.6-vicuna-13b-hf | DATASET: vqa_1k_control | loaded 1000
MODEL: llava-v1.6-vicuna-13b-hf | DATASET: vqa_1k_control_inst_blind | loaded 1000
MODEL: llava-1.5-7b-hf | DATASET: vqa_1k_control | loaded 1000
MODEL: llava-1.5-7b-hf | DATASET: vqa_1k_control_inst_blind | loaded 1000
MODEL: Qwen3-VL-4B-Instruct | DATASET: vqa_1k_control | loaded 1000
cannot process dict_keys(['image_id', 'question', 'question_id', 'deictic_removed', 'object_removed', 'weaker_object', 'subject_ablated', 'image', 'answers']) {'image_id': 524577, 'question': 'What number of clocks are on this tower?', 'question_id': 524577006, 'deictic_removed': 'What number of clocks are on the tower?', 'object_

model                     dataset                    control_type   
Qwen3-VL-4B-Instruct      vqa_1k_control_inst_blind  deictic_removed    1000
                                                     object_removed     1000
                                                     question           1000
                                                     subject_ablated    1000
                                                     weaker_object      1000
Qwen3-VL-8B-Instruct      vqa_1k_control             deictic_removed    1000
                                                     object_removed     1000
                                                     question           1000
                                                     subject_ablated    1000
                                                     weaker_object      1000
llava-1.5-7b-hf           vqa_1k_control             deictic_removed    1000
                                                     object_removed     1000
       

In [ ]:
df['confidence'] = df['confidence'].astype(str)
df.pivot_table(
    index=['model','question_id', 'control_type', ],
    columns=['dataset'],
    values=['question', 'output', 'confidence'],
    aggfunc=lambda x: ' | '.join(x)
).reindex(['question', 'deictic_removed', 'object_removed', 'weaker_object', 'subject_ablated'], level='control_type').dropna(axis=0) # .to_csv('./example_results_confidence.csv')

In [41]:
control_type_order = ['question', 'deictic_removed', 'object_removed', 'weaker_object', 'subject_ablated']
table = df.pivot_table(
    index=['control_type'],
    columns=['dataset', 'model'],
    values=['confidence'],
    aggfunc=['mean', 'count']
)
# Reindex the index to the desired order
table = table.reindex(control_type_order)
table.round(3)

mean                                           \
                          confidence                                            
dataset               vqa_1k_control                                            
model           Qwen3-VL-8B-Instruct llava-1.5-7b-hf llava-v1.6-mistral-7b-hf   
control_type                                                                    
question                       0.952           0.847                    0.910   
deictic_removed                0.952           0.847                    0.910   
object_removed                 0.944           0.822                    0.893   
weaker_object                  0.948           0.842                    0.904   
subject_ablated                0.946           0.819                    0.891   

                                                                  \
                                                                   
dataset                                                            
model           llava-v1.6-vicuna-13b-hf llava-v1.6-vicuna-7b-hf   
control_type                                                       
question                           0.875                   0.874   
deictic_removed                    0.874                   0.873   
object_removed                     0.847                   0.850   
weaker_object                      0.865                   0.867   
subject_ablated                    0.844                   0.846   

                                                           \
                                                            
dataset         vqa_1k_control_inst_blind                   
model                Qwen3-VL-4B-Instruct llava-1.5-7b-hf   
control_type                                                
question                            0.927           0.812   
deictic_removed                     0.926           0.812   
object_removed                      0.929           0.811   
weaker_object                       0.924           0.812   
subject_ablated                     0.930           0.812   

                                                                   \
                                                                    
dataset                                                             
model           llava-v1.6-mistral-7b-hf llava-v1.6-vicuna-13b-hf   
control_type                                                        
question                           0.865                    0.795   
deictic_removed                    0.866                    0.796   
object_removed                     0.863                    0.793   
weaker_object                      0.860                    0.788   
subject_ablated                    0.864                    0.793   

                                                       count                  \
                                                  confidence                   
dataset                                       vqa_1k_control                   
model           llava-v1.6-vicuna-7b-hf Qwen3-VL-8B-Instruct llava-1.5-7b-hf   
control_type                                                                   
question                          0.816                 1000            1000   
deictic_removed                   0.815                 1000            1000   
object_removed                    0.816                 1000            1000   
weaker_object                     0.815                 1000            1000   
subject_ablated                   0.819                 1000            1000   

                                                                   \
                                                                    
dataset                                                             
model           llava-v1.6-mistral-7b-hf llava-v1.6-vicuna-13b-hf   
control_type                                                        
question                            1000                     1000   
deictic_removed        

In [ ]:
df = analyze_confidences(results)
df.head()

,image_id,question_id,conf_question,pred_question,correct_question,acc_question,conf_deictic_removed,pred_deictic_removed,correct_deictic_removed,acc_deictic_removed,...,correct_object_removed,acc_object_removed,conf_weaker_object,pred_weaker_object,correct_weaker_object,acc_weaker_object,conf_subject_ablated,pred_subject_ablated,correct_subject_ablated,acc_subject_ablated
0,524577,524577006,0.999989,<|im_end|>,1,False,0.999986,<|im_end|>,1,False,...,1,False,0.976570,<|im_end|>,1,False,0.000000,2,12,False
1,524577,524577024,0.000000,building,on building,False,0.000000,building,on building,False,...,outside,False,0.733708,<|im_end|>,wall,False,0.665584,<|im_end|>,outside,False
2,524577,524577027,0.000000,abic,arabic,False,0.000000,abic,arabic,False,...,arabic,False,0.000000,abic,arabic,False,0.000000,abic,arabic,False
3,524799,524799000,0.360309,<|im_end|>,6,False,0.360309,<|im_end|>,6,False,...,young,False,0.393762,<|im_end|>,7,False,0.409214,<|im_end|>,young,False
4,524850,524850001,0.999949,<|im_end|>,2,False,0.999949,<|im_end|>,2,False,...,2,False,0.885704,<|im_end|>,1,False,0.657866,<|im_end|>,2,False
